In [ ]:
!pip install evaluate
!pip install tiktoken
!pip install SentencePiece
!pip install -U sentencepiece protobuf
!pip install scikit-learn

# --- Cell 1: Setup and Imports ---

In [1]:
import os
os.environ["HF_HOME"] = "/workspace/huggingface_cache"

import json
import torch
import numpy as np
from datasets import Dataset
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification, 
    TrainingArguments, 
    Trainer,
    DataCollatorWithPadding
)
import evaluate

print(f"CUDA Available: {torch.cuda.is_available()}")

CUDA Available: True


# --- Cell 2: Load and Format Data ---

In [2]:
# These files should contain the rationales generated by your Phase 3 DPO model
# TRAIN_FILE = "train_with_rationales.json"
# VAL_FILE = "validation_with_rationales.json"

TRAIN_FILE = "train_with_SFT_rationales.json"
VAL_FILE = "val_with_SFT_rationales.json"
CACHE_FILE = "grounding_cache.json"

cache = json.load(open(CACHE_FILE, 'r', encoding='utf-8'))

def prepare_classification_dataset(filepath):
    raw_data = json.load(open(filepath, 'r', encoding='utf-8'))
    data_rows = []
    
    for item in raw_data:
        img_path = item.get("local_image_path", "").replace("\\\\", "/").replace("\\", "/")
        if img_path not in cache or not cache[img_path].get("deplot_table"):
            continue
            
        desc = item.get("title_description", "")
        # Truncate description to prevent token bloat
        if desc:
            words = desc.split()
            if len(words) > 50:
                desc = " ".join(words[:50]) + "..."
                
        # DeBERTa works exceptionally well with two sequences separated by a [SEP] token.
        # Sequence 1: The Grounding Context + Claim
        # Sequence 2: The Generated Rationale
        
        context_str = (
            f"Title: {desc}\n"
            f"Table: {cache[img_path]['deplot_table']}\n"
            f"Spec: {cache[img_path]['vega_lite_spec']}\n"
            f"Claim: {item['claim']}"
        )
        
        data_rows.append({
            "context": context_str,
            "rationale": item["generated_rationale"], # From your Phase 3 model
            "label": 1 if item["label"] else 0 # Explicit binary int conversion
        })
        
    return Dataset.from_list(data_rows)

train_dataset = prepare_classification_dataset(TRAIN_FILE)
val_dataset = prepare_classification_dataset(VAL_FILE)
print(f"Loaded {len(train_dataset)} training samples and {len(val_dataset)} validation samples.")

Loaded 7607 training samples and 953 validation samples.


# --- Cell 3: Tokenization ---

In [3]:
MODEL_ID = "microsoft/deberta-v3-base" # Or 'large' if you want maximum accuracy
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

def tokenize_function(examples):
    # Passing two separate strings to the tokenizer automatically inserts the 
    # [SEP] token between them, allowing DeBERTa to perform cross-attention properly.
    return tokenizer(
        examples["context"],
        examples["rationale"],
        truncation='only_first',
        max_length=2048, # DeBERTa-v3 can handle long sequences, adjust if VRAM is tight
        force_download=True
    )

tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_val = val_dataset.map(tokenize_function, batched=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

Map:   0%|          | 0/7607 [00:00<?, ? examples/s]

Map:   0%|          | 0/953 [00:00<?, ? examples/s]

# --- Cell 4: Model and Metrics ---

In [4]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL_ID, num_labels=2)

accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    # The model outputs raw logits. We take the argmax to get the predicted class (0 or 1)
    predictions = np.argmax(predictions, axis=1)
    
    acc = accuracy_metric.compute(predictions=predictions, references=labels)
    f1 = f1_metric.compute(predictions=predictions, references=labels)
    
    return {"accuracy": acc["accuracy"], "f1": f1["f1"]}

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
pooler.dense.weight                     | MISSING    | 
classifier.bias                         | MISSING    | 
pooler.dense.bias                       | MISSING    | 
classifier.weight        

# --- Cell 5: Training Setup and Execution *Change Output Dir for DPO--- 

In [5]:
training_args = TrainingArguments(
    output_dir="/workspace/deberta-chart-verifier-SFT", 
    learning_rate=2e-5, 
    
    # --- VRAM OPTIMIZATIONS ---
    per_device_train_batch_size=4,   # Reduced from 8
    gradient_accumulation_steps=2,   # 4 x 2 = Global Batch Size of 8 (Same as before)
    per_device_eval_batch_size=4,    # Reduced from 8
    gradient_checkpointing=True,     # 🚀 CRITICAL: Saves massive VRAM on 2048 token lengths
    
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    fp16=False,
    bf16=True,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("Starting DeBERTa Verification Training...")
trainer.train()

metrics = trainer.evaluate()
print(f"Final Validation Accuracy: {metrics['eval_accuracy']:.4f}")

trainer.save_model("/workspace/deberta-chart-verifier-final")
print("Training Complete. Model saved.")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 1}.


Starting DeBERTa Verification Training...


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,1.842028,0.699750,0.481637,0.000000
2,1.398630,0.692529,0.518363,0.682792
3,1.388736,0.692699,0.518363,0.682792


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['deberta.embeddings.LayerNorm.weight', 'deberta.embeddings.LayerNorm.bias', 'deberta.encoder.layer.0.attention.output.LayerNorm.weight', 'deberta.encoder.layer.0.attention.output.LayerNorm.bias', 'deberta.encoder.layer.0.output.LayerNorm.weight', 'deberta.encoder.layer.0.output.LayerNorm.bias', 'deberta.encoder.layer.1.attention.output.LayerNorm.weight', 'deberta.encoder.layer.1.attention.output.LayerNorm.bias', 'deberta.encoder.layer.1.output.LayerNorm.weight', 'deberta.encoder.layer.1.output.LayerNorm.bias', 'deberta.encoder.layer.2.attention.output.LayerNorm.weight', 'deberta.encoder.layer.2.attention.output.LayerNorm.bias', 'deberta.encoder.layer.2.output.LayerNorm.weight', 'deberta.encoder.layer.2.output.LayerNorm.bias', 'deberta.encoder.layer.3.attention.output.LayerNorm.weight', 'deberta.encoder.layer.3.attention.output.LayerNorm.bias', 'deberta.encoder.layer.3.output.LayerNorm.weight', 'deberta.encoder.layer.3.output.Laye

RuntimeError: on_train_begin must be called before on_evaluate